<a href="https://colab.research.google.com/github/panwarchirag2006-rgb/CHIRAGPANWAR.2026B03110265/blob/main/Food_Delivery_Operations_Analytics_Chirag_Panwar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 8: Food Delivery Operations Analytics

**Student Name:** Chirag Panwar  
**Roll Number:** 2026B03110265  
**Industry:** FoodTech / Logistics  
**Dataset:** `08_food_delivery.csv`  

## Project overview
This notebook analyzes food-delivery orders to evaluate delivery times, order outcomes, restaurant performance, peak ordering hours, city-level operations and revenue patterns. It completes assignment tasks **109–123**.

**Tools:** Python, Pandas and NumPy.  
**Visuals:** No graphs or charts are used; results are presented as tables and printed summaries.

## How to run in Google Colab

## 1. Import libraries and load dataset (Task 109)

In [3]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_FILE = "08_food_delivery_updated-1.csv"

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            DATA_FILE = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            f"Place {DATA_FILE} in the same folder as this notebook, then rerun."
        )

df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nFirst five records:")
display(df.head())
print("\nDataset information:")
df.info()
print("\nNumerical summary:")
display(df.describe())


Dataset loaded successfully
Rows: 600 | Columns: 9

Column names:
['Order_ID', 'City', 'Restaurant', 'Order_Value', 'Distance_km', 'Delivery_Time_Min', 'Order_Status', 'Payment_Mode', 'Order_Hour']

Data types:
Order_ID              object
City                  object
Restaurant            object
Order_Value          float64
Distance_km          float64
Delivery_Time_Min    float64
Order_Status          object
Payment_Mode          object
Order_Hour             int64
dtype: object

First five records:


,Order_ID,City,Restaurant,Order_Value,Distance_km,Delivery_Time_Min,Order_Status,Payment_Mode,Order_Hour
0,FD00000,Noida,Green Bowl,588.00,2.60,32.00,Delivered,Card,10
1,FD00001,Delhi,Pizza Point,496.68,5.30,36.00,Delivered,Cash,22
2,FD00002,Faridabad,Pizza Point,419.41,6.00,42.00,Delivered,Cash,18
3,FD00003,Noida,Pizza Point,195.48,5.00,40.00,Delivered,UPI,19
4,FD00004,Faridabad,Cafe 24,621.70,2.50,15.00,Delivered,Cash,22



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order_ID           600 non-null    object 
 1   City               600 non-null    object 
 2   Restaurant         600 non-null    object 
 3   Order_Value        600 non-null    float64
 4   Distance_km        600 non-null    float64
 5   Delivery_Time_Min  593 non-null    float64
 6   Order_Status       600 non-null    object 
 7   Payment_Mode       600 non-null    object 
 8   Order_Hour         600 non-null    int64  
dtypes: float64(3), int64(1), object(5)
memory usage: 42.3+ KB

Numerical summary:


,Order_Value,Distance_km,Delivery_Time_Min,Order_Hour
count,600.00,600.00,593.00,600.00
mean,513.88,5.52,37.76,14.77
std,254.93,2.68,13.10,5.40
min,80.00,0.50,10.00,0.00
25%,355.62,3.80,29.00,10.00
50%,474.54,5.30,36.00,15.00
75%,618.80,6.72,46.00,19.00
max,"1,489.33",14.60,75.00,23.00


## 2. Validate data and inspect missing delivery times (Task 110)

In [4]:
required_columns = [
    "Order_ID", "City", "Restaurant", "Order_Value", "Distance_km",
    "Delivery_Time_Min", "Order_Status", "Payment_Mode", "Order_Hour"
]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print("Missing values by column:")
display(df.isna().sum().rename("Missing_Count").to_frame())

missing_delivery_count = df["Delivery_Time_Min"].isna().sum()
missing_delivery_pct = df["Delivery_Time_Min"].isna().mean() * 100
print(f"Missing delivery times: {missing_delivery_count} ({missing_delivery_pct:.2f}%)")

# Delivery time is meaningful for completed deliveries. Use the median of
# non-missing delivered-order times to fill missing values for analysis.
delivered_times = df.loc[
    (df["Order_Status"].eq("Delivered")) & df["Delivery_Time_Min"].notna(),
    "Delivery_Time_Min"
]
if delivered_times.empty:
    delivered_times = df["Delivery_Time_Min"].dropna()

if missing_delivery_count:
    fill_value = delivered_times.median()
    df["Delivery_Time_Min"] = df["Delivery_Time_Min"].fillna(fill_value)
    print(f"Filled missing delivery times with delivered-order median: {fill_value:.2f} minutes")
else:
    print("No missing delivery times required treatment.")

print("Missing delivery times after handling:", df["Delivery_Time_Min"].isna().sum())


Missing values by column:


,Missing_Count
Order_ID,0
City,0
Restaurant,0
Order_Value,0
Distance_km,0
Delivery_Time_Min,7
Order_Status,0
Payment_Mode,0
Order_Hour,0


Missing delivery times: 7 (1.17%)
Filled missing delivery times with delivered-order median: 36.00 minutes
Missing delivery times after handling: 0


## 3. Check duplicate Order_ID values (Task 111)

In [5]:
duplicate_mask = df["Order_ID"].duplicated(keep=False)
duplicate_ids = df.loc[duplicate_mask, "Order_ID"].sort_values()
print("Number of duplicated Order_ID rows:", int(duplicate_mask.sum()))
print("Number of duplicated Order_ID values:", int(df["Order_ID"].duplicated().sum()))
if duplicate_mask.any():
    display(df.loc[duplicate_mask].sort_values("Order_ID"))
else:
    print("No duplicate Order_ID values found.")

# Keep the original records for auditability. If duplicates are discovered,
# review them before removing because repeated IDs may represent data issues.


Number of duplicated Order_ID rows: 0
Number of duplicated Order_ID values: 0
No duplicate Order_ID values found.


## 4. Total orders and order-status percentages (Task 112)

In [6]:
total_orders = len(df)
status_summary = (
    df["Order_Status"].value_counts()
      .rename_axis("Order_Status")
      .to_frame("Order_Count")
)
status_summary["Percentage"] = status_summary["Order_Count"] / total_orders * 100
print("Total orders:", total_orders)
display(status_summary.round(2))


Total orders: 600


,Order_Count,Percentage
Order_Status,,
Delivered,436,72.67
Delayed,87,14.50
Cancelled,77,12.83


## 5. Average delivery time by city (Task 113)

In [7]:
city_delivery = (
    df.groupby("City")
      .agg(
          Average_Delivery_Time_Min=("Delivery_Time_Min", "mean"),
          Median_Delivery_Time_Min=("Delivery_Time_Min", "median"),
          Orders=("Order_ID", "count")
      )
      .sort_values("Average_Delivery_Time_Min", ascending=False)
)
display(city_delivery.round(2))


,Average_Delivery_Time_Min,Median_Delivery_Time_Min,Orders
City,,,
Noida,38.22,36.00,147
Gurugram,38.08,36.00,139
Delhi,37.69,37.00,148
Faridabad,37.09,36.00,166


## 6. Five restaurants with highest total order value (Task 114)

In [8]:
restaurant_revenue = (
    df.groupby("Restaurant")["Order_Value"]
      .sum()
      .sort_values(ascending=False)
      .rename("Total_Order_Value")
      .to_frame()
)
top_five_restaurants = restaurant_revenue.head(5)
display(top_five_restaurants.round(2))


,Total_Order_Value
Restaurant,
Green Bowl,"73,953.32"
Urban Bites,"63,207.24"
Pizza Point,"61,212.93"
Spice Hub,"55,068.74"
Cafe 24,"54,886.07"


## 7. Average order value by restaurant (Task 115)

In [9]:
restaurant_aov = (
    df.groupby("Restaurant")
      .agg(
          Average_Order_Value=("Order_Value", "mean"),
          Total_Orders=("Order_ID", "count"),
          Total_Order_Value=("Order_Value", "sum")
      )
      .sort_values("Average_Order_Value", ascending=False)
)
display(restaurant_aov.round(2))


,Average_Order_Value,Total_Orders,Total_Order_Value
Restaurant,,,
Green Bowl,551.89,134,"73,953.32"
Urban Bites,513.88,123,"63,207.24"
Cafe 24,512.95,107,"54,886.07"
Spice Hub,509.90,108,"55,068.74"
Pizza Point,478.23,128,"61,212.93"


## 8. Identify peak ordering hours (Task 116)

In [10]:
hourly_orders = (
    df.groupby("Order_Hour")
      .size()
      .rename("Order_Count")
      .sort_values(ascending=False)
      .to_frame()
)
hourly_orders["Share_Percent"] = hourly_orders["Order_Count"] / total_orders * 100
peak_hour = int(hourly_orders.index[0])
peak_count = int(hourly_orders.iloc[0]["Order_Count"])
print(f"Peak ordering hour: {peak_hour:02d}:00–{(peak_hour + 1) % 24:02d}:00 ({peak_count} orders)")
display(hourly_orders.sort_values("Order_Hour") if False else hourly_orders)


Peak ordering hour: 13:00–14:00 (43 orders)


,Order_Count,Share_Percent
Order_Hour,,
13,43,7.17
10,42,7.00
18,42,7.00
21,40,6.67
20,38,6.33
15,37,6.17
11,37,6.17
8,36,6.00
9,36,6.00


## 9. Compare delivery time across order-status categories (Task 117)

In [11]:
status_delivery = (
    df.groupby("Order_Status")
      .agg(
          Average_Delivery_Time_Min=("Delivery_Time_Min", "mean"),
          Median_Delivery_Time_Min=("Delivery_Time_Min", "median"),
          Orders=("Order_ID", "count")
      )
      .sort_values("Average_Delivery_Time_Min", ascending=False)
)
print("Note: Missing delivery times were median-imputed in Task 110. Interpret")
print("delivery-time averages for cancelled orders cautiously, because a cancelled")
print("order may not have a completed delivery duration.")
display(status_delivery.round(2))


Note: Missing delivery times were median-imputed in Task 110. Interpret
delivery-time averages for cancelled orders cautiously, because a cancelled
order may not have a completed delivery duration.


,Average_Delivery_Time_Min,Median_Delivery_Time_Min,Orders
Order_Status,,,
Delayed,40.66,40.00,87
Cancelled,39.83,40.00,77
Delivered,36.79,36.00,436


## 10. Cancellation and delay rates by city (Task 118)

In [12]:
city_outcomes = (
    df.assign(
        Is_Cancelled=df["Order_Status"].eq("Cancelled"),
        Is_Delayed=df["Order_Status"].eq("Delayed")
    )
    .groupby("City")
    .agg(
        Total_Orders=("Order_ID", "count"),
        Cancelled_Orders=("Is_Cancelled", "sum"),
        Delayed_Orders=("Is_Delayed", "sum")
    )
)
city_outcomes["Cancellation_Rate_Percent"] = (
    city_outcomes["Cancelled_Orders"] / city_outcomes["Total_Orders"] * 100
)
city_outcomes["Delay_Rate_Percent"] = (
    city_outcomes["Delayed_Orders"] / city_outcomes["Total_Orders"] * 100
)
display(city_outcomes.round(2))


,Total_Orders,Cancelled_Orders,Delayed_Orders,Cancellation_Rate_Percent,Delay_Rate_Percent
City,,,,,
Delhi,148,22,23,14.86,15.54
Faridabad,166,21,18,12.65,10.84
Gurugram,139,18,23,12.95,16.55
Noida,147,16,23,10.88,15.65


## 11. Median and standard deviation using NumPy (Task 119)

In [13]:
delivery_values = df["Delivery_Time_Min"].dropna().to_numpy()
delivery_median = np.median(delivery_values)
delivery_std_population = np.std(delivery_values)       # ddof=0: population SD
delivery_std_sample = np.std(delivery_values, ddof=1)   # ddof=1: sample SD

print(f"Median delivery time: {delivery_median:.2f} minutes")
print(f"Population standard deviation: {delivery_std_population:.2f} minutes")
print(f"Sample standard deviation: {delivery_std_sample:.2f} minutes")


Median delivery time: 36.00 minutes
Population standard deviation: 13.02 minutes
Sample standard deviation: 13.03 minutes


## 12. Identify unusually long deliveries using IQR (Task 120)

In [14]:
q1 = df["Delivery_Time_Min"].quantile(0.25)
q3 = df["Delivery_Time_Min"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

long_delivery_outliers = df.loc[
    df["Delivery_Time_Min"] > upper_bound,
    ["Order_ID", "City", "Restaurant", "Delivery_Time_Min", "Order_Status"]
].sort_values("Delivery_Time_Min", ascending=False)

print(f"Q1: {q1:.2f} | Q3: {q3:.2f} | IQR: {iqr:.2f}")
print(f"Upper IQR threshold: {upper_bound:.2f} minutes")
print("Unusually long delivery records:", len(long_delivery_outliers))
display(long_delivery_outliers)


Q1: 29.00 | Q3: 46.00 | IQR: 17.00
Upper IQR threshold: 71.50 minutes
Unusually long delivery records: 7


,Order_ID,City,Restaurant,Delivery_Time_Min,Order_Status
454,FD00454,Noida,Spice Hub,75.00,Delivered
526,FD00526,Gurugram,Spice Hub,74.00,Delayed
557,FD00557,Noida,Urban Bites,74.00,Delayed
566,FD00566,Faridabad,Urban Bites,74.00,Cancelled
548,FD00548,Delhi,Urban Bites,72.00,Cancelled
583,FD00583,Gurugram,Urban Bites,72.00,Delivered
585,FD00585,Noida,Pizza Point,72.00,Delivered


## 13. Create delivery-time bands (Task 121)

In [15]:
# Operational bands used for this assignment:
# Fast: <= 30 minutes; Standard: 31–45 minutes; Delayed: > 45 minutes.
band_labels = ["Fast", "Standard", "Delayed"]
df["Delivery_Time_Band"] = pd.cut(
    df["Delivery_Time_Min"],
    bins=[-np.inf, 30, 45, np.inf],
    labels=band_labels,
    right=True
)

delivery_band_summary = (
    df["Delivery_Time_Band"].value_counts()
      .reindex(band_labels, fill_value=0)
      .rename_axis("Delivery_Time_Band")
      .to_frame("Order_Count")
)
delivery_band_summary["Percentage"] = (
    delivery_band_summary["Order_Count"] / total_orders * 100
)
display(delivery_band_summary.round(2))


,Order_Count,Percentage
Delivery_Time_Band,,
Fast,171,28.50
Standard,275,45.83
Delayed,154,25.67


## 14. Group revenue by city and payment mode (Task 122)

In [16]:
revenue_city_payment = pd.pivot_table(
    df,
    values="Order_Value",
    index="City",
    columns="Payment_Mode",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)
print("Revenue (sum of Order_Value) by city and payment mode:")
display(revenue_city_payment.round(2))


Revenue (sum of Order_Value) by city and payment mode:


Payment_Mode,Card,Cash,UPI,Total
City,,,,
Delhi,"21,614.92","28,004.51","29,072.44","78,691.87"
Faridabad,"39,617.53","23,737.94","18,437.28","81,792.75"
Gurugram,"23,729.13","28,309.23","22,369.43","74,407.79"
Noida,"25,925.92","23,278.80","24,231.17","73,435.89"
Total,"110,887.50","103,330.48","94,110.32","308,328.30"


## 15. Five operational recommendations (Task 123)

In [17]:
# Generate recommendations from the measured results.
slowest_city = city_delivery["Average_Delivery_Time_Min"].idxmax()
slowest_city_time = city_delivery.loc[slowest_city, "Average_Delivery_Time_Min"]
highest_cancel_city = city_outcomes["Cancellation_Rate_Percent"].idxmax()
highest_cancel_rate = city_outcomes.loc[highest_cancel_city, "Cancellation_Rate_Percent"]
highest_delay_city = city_outcomes["Delay_Rate_Percent"].idxmax()
highest_delay_rate = city_outcomes.loc[highest_delay_city, "Delay_Rate_Percent"]
top_restaurant = restaurant_revenue.index[0]
top_restaurant_value = restaurant_revenue.iloc[0, 0]
peak_hour_text = f"{peak_hour:02d}:00–{(peak_hour + 1) % 24:02d}:00"

recommendations = [
    f"Prioritize delivery-process review in {slowest_city}, which has the highest average delivery time ({slowest_city_time:.2f} minutes). Review rider allocation, dispatch waiting time and route planning.",
    f"Investigate cancellation causes in {highest_cancel_city}, where the cancellation rate is highest ({highest_cancel_rate:.2f}%). Review stock availability, restaurant acceptance and customer communication.",
    f"Focus delay-reduction actions in {highest_delay_city}, which has the highest delayed-order rate ({highest_delay_rate:.2f}%). Track preparation time and rider assignment separately.",
    f"Plan staffing and rider capacity around the busiest observed hour ({peak_hour_text}); use hourly demand counts to avoid over- or under-staffing.",
    f"Maintain service quality and monitor capacity at {top_restaurant}, the highest total-order-value restaurant (order value total: {top_restaurant_value:,.2f}). Use restaurant-level average order value alongside order volume when planning partnerships."
]

for i, recommendation in enumerate(recommendations, start=1):
    print(f"{i}. {recommendation}")


1. Prioritize delivery-process review in Noida, which has the highest average delivery time (38.22 minutes). Review rider allocation, dispatch waiting time and route planning.
2. Investigate cancellation causes in Delhi, where the cancellation rate is highest (14.86%). Review stock availability, restaurant acceptance and customer communication.
3. Focus delay-reduction actions in Gurugram, which has the highest delayed-order rate (16.55%). Track preparation time and rider assignment separately.
4. Plan staffing and rider capacity around the busiest observed hour (13:00–14:00); use hourly demand counts to avoid over- or under-staffing.
5. Maintain service quality and monitor capacity at Green Bowl, the highest total-order-value restaurant (order value total: 73,953.32). Use restaurant-level average order value alongside order volume when planning partnerships.


## 16. Final consolidated output / project conclusion

The tables above provide the requested city performance, restaurant ranking, peak-hour summary, delay analysis and revenue-by-payment-mode output. The results are calculated from the supplied dataset each time the notebook runs.

### Data interpretation notes
- The source file contains 600 order records and 9 columns.
- The seven missing delivery-time values are handled using the median of available delivered-order times (falling back to the overall available median if necessary).
- Duplicate Order_ID values are checked and displayed; records are not silently deleted.
- Restaurant total order value and revenue pivot tables sum `Order_Value` across all order statuses. This represents recorded order value, not necessarily realized revenue from completed orders.
- Delivery-time bands (Fast ≤30, Standard 31–45, Delayed >45 minutes) are operational thresholds selected for this analysis, not source-provided categories.
- The dataset does not specify a currency, so order values are reported in the dataset's original units.
- Imputed delivery times are included in descriptive delivery-time summaries; cancelled-order duration comparisons should be interpreted cautiously.

**End of Project 8 – Food Delivery Operations Analytics.**
